In [115]:
import pandas as pds
import os, shutil, math
import numpy as np

from array import *
def array_list(array_num):
    num_list = array_num.tolist() # list
  


In [117]:
parent_dir = '/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/MOFs/'
path_General_Files = '/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/General Files/'

### UFF Parameters
LJ_UFF = ('/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/General Files/UFF_data.xlsx')
LJ_data = pds.read_excel(LJ_UFF, 'LJ' )
ELEMENT = np.array(LJ_data['Element'])
ELEMENT = ELEMENT.tolist()

EPSILON = np.array(LJ_data['Energy(kcal/mol) '])
EPSILON = EPSILON.tolist()

SIGMA = np.array(LJ_data['sigma'])
SIGMA = SIGMA.tolist()

In [120]:
for folder in os.listdir(parent_dir):
    path_MOF = os.path.join(parent_dir, folder)
    print(path_MOF)
    
    #### Creating npt.in file 
    
    for filename in os.listdir(path_MOF):
        shutil.copy2(os.path.join(path_General_Files,'npt.in'),path_MOF)
        if filename == 'data.'+str(folder):
            N_O = None 
            N_H = None
            b_W = None
            a_W = None
            F_n = None 
            F_b = None 
            F_a = None
            F_d = None
            F_i = None 
            N_F = None 
            atom_types = 0
            bonds_types = 0
            angle_types = 0
            dihedral_types = 0
            improper_types = 0
            H_defect = []
            
            epsilon = []
            sigma = []
            
            
            path_data_MOF = os.path.join(path_MOF, filename)
            fr = open(str(path_data_MOF), "rt")
            lines = fr.readlines()
            for line in lines:
                row=line.split()
                if not line.strip():
                    continue
                    
                if len(row) == 7 and row[1] == '444' and row[2] == '2' and 0.3 < float(row[3]) < 0.5:
                    H_defect.append(row[0])
                    H_defect_atoms = ' '.join(H_defect)
                    
                if len(row) == 2 and row[1] == 'atoms':
                    N_F = int(row[0])
                if len(row) == 4 and row[2] == '#':
                    atom_types = atom_types + 1
                if len(row) == 7 and row[1] == 'harmonic':
                    bonds_types = bonds_types + 1
                if len(row) == 10 and row[1] =='fourier':
                    angle_types = angle_types + 1
                if len(row) == 9 and row[1] =='cosine/periodic':
                    angle_types = angle_types + 1
                if len(row) == 10 and row[1] =='harmonic':
                    dihedral_types = dihedral_types  + 1   
                if len(row) == 12 and row[1] =='fourier':
                    improper_types = improper_types + 1           
                if len(row) == 12 and row[1] =='fourier':
                    improper_types = improper_types + 1 
                if len(row) == 4 and row[2] == '#':
                    a_string = str(row[3])
                    partitioned_string = a_string.partition('_')
                    Element = partitioned_string[0]
                    
                    if len(Element) > 1:
                        Z = ELEMENT.index(Element[:2])
                    if len(Element) == 1:
                        Z = ELEMENT.index(Element)
                        
                        
                    epsilon.append(float(EPSILON[Z]))
                    sigma.append(float(SIGMA[Z]))
            
            N_O = atom_types + 1
            N_H = N_O + 1
            b_W = bonds_types + 1
            a_W = angle_types + 1
            F_n = atom_types
            F_b = bonds_types
            F_a = angle_types
            F_d = dihedral_types
            F_i = improper_types
            
            epsilon.append(float(0.1553))
            sigma.append(float(3.1660))
            
            epsilon.append(float(0.0000))
            sigma.append(float(1.0000))   

            print(epsilon)
            print(sigma)
    for file in os.listdir(path_MOF):
        if file == 'data.adsorbate':
            N_W_i = None
            N_W_f = None 
            path_data_adsorbate = os.path.join(path_MOF, file)
            fr = open(str(path_data_adsorbate), "rt")
            lines = fr.readlines()
            for line in lines:
                row=line.split()
                if not line.strip():
                    continue
                if len(row) == 2 and row[1] == 'atoms':
                    N_W_i = N_F + 1
                    N_W_f = N_W_i + int(row[0])
                    
     
    fin = open(''+str(path_MOF)+'/npt.in', "rt")

    data = fin.read()
    data = data.replace('N_O', str(N_O))
    data = data.replace('N_H', str(N_H))
    data = data.replace('b_W', str(b_W))
    data = data.replace('a_W', str(a_W))
    data = data.replace('F_n', str(F_n))
    data = data.replace('F_b', str(F_b))
    data = data.replace('F_a', str(F_a))
    data = data.replace('F_d', str(F_d))
    data = data.replace('F_i', str(F_i))
    data = data.replace('MOF_Name', str(folder))
    data = data.replace('N_F', str(N_F))
    data = data.replace('N_W_i', str(N_W_i))
    data = data.replace('N_W_f', str(N_W_f))
    data = data.replace('H_defect_atoms', str(H_defect_atoms))

    fin.close()

    fin = open(''+str(path_MOF)+'/npt.in', "wt")

    fin.write(data)
    
    #### Creating data.pair file 
    
    path_data_pair = os.path.join(path_MOF,'data.pair')
    
    with open(str(path_data_pair), 'a') as file:
        for i in range(1, int(len(epsilon)) + 1):
            for j in range(i, int(len(epsilon)) + 1):
                pair_epsilon = float(math.sqrt(epsilon[i-1]*epsilon[j-1]))
                pair_sigma = float(sigma[i-1]+sigma[j-1])/2
                pair_line = 'pair_coeff' + '    ' + str(i) + '    ' + str(j) + '    ' +  'lj/cut/tip4p/long' + '    ' + str(pair_epsilon) + '    ' + str(pair_sigma) + os.linesep 
                file.write(str(pair_line))
    

/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/MOFs/IDIWOH_111
[0.016, 0.044, 0.105, 0.06, 0.06, 0.06, 0.1553, 0.0]
[2.8009855698332267, 2.5711337005530193, 3.4308509635584463, 3.1181455134911875, 3.1181455134911875, 3.1181455134911875, 3.166, 1.0]
/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/MOFs/LASYOU_111
[0.005, 0.044, 0.105, 0.06, 0.06, 0.1553, 0.0]
[3.113691019900486, 2.5711337005530193, 3.4308509635584463, 3.1181455134911875, 3.1181455134911875, 3.166, 1.0]
/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/MOFs/MIBQAR_121
[0.124, 0.044, 0.105, 0.06, 0.06, 0.06, 0.1553, 0.0]
[2.4615531582217574, 2.5711337005530193, 3.4308509635584463, 3.1181455134911875, 3.1181455134911875, 3.1181455134911875, 3.166, 1.0]
/Users/shubhamjamdade/Desktop/MD Snapshot/Convert CIF to data.adsorbate/MOFs/ATOXEN_111
[0.124, 0.044, 0.105, 0.105, 0.069, 0.06, 0.06, 0.1553, 0.0]
[2.4615531582217574, 2.5711337005530193, 3.4308509635584